# Infrastructure and DataOps

Notebook for **Chapter 12** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch12.html

Tip: change a number, predict the output, then run the cell again.

## 12.1 Docker and Kubernetes Basics

## 12.2 Infrastructure as Code, CI/CD and Secrets

## 12.3 Monitoring, Alerting, Runbooks and Cost

In [ ]:
import datetime, statistics as st
now=datetime.datetime(2026,1,10,9,0)
last=datetime.datetime(2026,1,10,6,30); sla=datetime.timedelta(hours=2)
print("freshness lag:",now-last,"breach:",now-last>sla)
hist=[10200,9900,10100,10050,9950,10150,10000]; today=6100
m=st.mean(hist); s=st.stdev(hist); z=(today-m)/s
print(f"volume mean={m:.0f} sd={s:.1f} today={today} z={z:.1f} alert={abs(z)>3}")

In [ ]:
tb_scanned=2.5; price=6.25  # illustrative price per TB scanned, NOT a quote
print("query cost:",tb_scanned*price,"per run; 30 runs:",tb_scanned*price*30)
with_part=tb_scanned*0.1; print("after partition pruning (10% scanned):",with_part*price*30)

query cost: 15.625 per run; 30 runs: 468.75
after partition pruning (10% scanned): 46.875


## 12.4 Governance, Privacy, PII, Lineage and Catalogs

In [ ]:
import hashlib
phone="9876543210"; email="asha.rao@example.com"
print("masked phone:",phone[:2]+"*"*6+phone[-2:])
user,dom=email.split("@"); print("masked email:",user[0]+"***@"+dom)
salt="demo-salt-not-a-secret"
tok=lambda s: hashlib.sha256((salt+s).encode()).hexdigest()[:12]
print("token:",tok(email),"same input same token:",tok(email)==tok(email))

masked phone: 98******10
masked email: a***@example.com
token: e841b43a3067 same input same token: True


## 12.5 Security and Privacy in Depth: Access Policies, Tokenization and Erasure

Section 12.4 introduced PII, lineage and catalogs. This section goes one level deeper into the three things a data engineer is actually asked to build: control who sees what, make personal data safe to use, and delete a person's data when the law says so. All code here runs. The cloud-specific syntax (Snowflake masking policies, BigQuery row access policies, Unity Catalog grants) differs by vendor, so we model the logic and tell you where to look up the syntax.

In [ ]:
# RBAC says WHO may read a table. Row and column policies say WHAT they see inside it.
rows = [
  {"order_id": 1, "region": "IN", "email": "asha@example.com", "card_last4": "4242", "amount": 120},
  {"order_id": 2, "region": "IN", "email": "ravi@example.com", "card_last4": "1111", "amount": 80},
  {"order_id": 3, "region": "DE", "email": "jana@example.com", "card_last4": "0005", "amount": 200},
]
roles = {
  "analyst_in": {"regions": {"IN"}, "see_pii": False},
  "support_de": {"regions": {"DE"}, "see_pii": True},
  "admin":      {"regions": {"IN", "DE"}, "see_pii": True},
}
def mask_email(e):
    user, domain = e.split("@")
    return user[0] + "***@" + domain
def read(role):
    p = roles[role]
    out = []
    for r in rows:
        if r["region"] not in p["regions"]:           # row-level policy
            continue
        r = dict(r)
        if not p["see_pii"]:                          # column masking policy
            r["email"] = mask_email(r["email"]); r["card_last4"] = "****"
        out.append(r)
    return out
for role in roles:
    print(role, "->", [(r["order_id"], r["email"], r["card_last4"]) for r in read(role)])

analyst_in -> [(1, 'a***@example.com', '****'), (2, 'r***@example.com', '****')]
support_de -> [(3, 'jana@example.com', '0005')]
admin -> [(1, 'asha@example.com', '4242'), (2, 'ravi@example.com', '1111'), (3, 'jana@example.com', '0005')]


In [ ]:
# Tokenization and pseudonymisation: keep joins working without keeping the raw value.
import hmac, hashlib
SECRET = b"demo-secret-kept-in-a-secrets-manager"   # never in code in real life
def token(value: str) -> str:
    return hmac.new(SECRET, value.lower().encode(), hashlib.sha256).hexdigest()[:12]
orders = [("asha@example.com", 120), ("ravi@example.com", 80), ("Asha@Example.com", 45)]
tok = [(token(e), a) for e, a in orders]
print(tok)
totals = {}
for t, a in tok: totals[t] = totals.get(t, 0) + a
print("spend per customer token:", totals)
# plain unsalted hash is guessable: an attacker hashes a list of known emails and compares.
plain = hashlib.sha256(b"asha@example.com").hexdigest()[:12]
guess = {hashlib.sha256(e.encode()).hexdigest()[:12]: e for e in ["ravi@example.com", "asha@example.com"]}
print("plain sha256 reversed by guessing:", guess.get(plain))
guess2 = {hashlib.sha256(e.encode()).hexdigest()[:12]: e for e in ["ravi@example.com", "asha@example.com"]}
print("keyed token reversed by guessing:", guess2.get(token("asha@example.com")))

[('aa9265b6289e', 120), ('6932565bea37', 80), ('aa9265b6289e', 45)]
spend per customer token: {'aa9265b6289e': 165, '6932565bea37': 80}
plain sha256 reversed by guessing: asha@example.com
keyed token reversed by guessing: None


In [ ]:
# Right to erasure across a medallion pipeline, using sqlite as the stand-in for tables.
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE bronze_events(id INTEGER, user_id TEXT, payload TEXT);
CREATE TABLE silver_orders(order_id INTEGER, user_id TEXT, amount REAL);
CREATE TABLE gold_daily(day TEXT, revenue REAL, orders INTEGER);
CREATE TABLE erasure_log(user_id TEXT, table_name TEXT, rows_deleted INTEGER);
INSERT INTO bronze_events VALUES (1,'u7','{"page":"home"}'),(2,'u7','{"page":"cart"}'),(3,'u8','{"page":"home"}');
INSERT INTO silver_orders VALUES (10,'u7',50),(11,'u8',70),(12,'u7',30);
INSERT INTO gold_daily VALUES ('2025-01-01',150,3);
""")
def erase(user):
    for table in ("bronze_events", "silver_orders"):
        n = db.execute(f"DELETE FROM {table} WHERE user_id=?", (user,)).rowcount
        db.execute("INSERT INTO erasure_log VALUES (?,?,?)", (user, table, n))
    db.commit()
erase("u7")
print("bronze left:", db.execute("SELECT COUNT(*) FROM bronze_events").fetchone()[0])
print("silver left:", db.execute("SELECT order_id, user_id FROM silver_orders").fetchall())
print("log:", db.execute("SELECT * FROM erasure_log").fetchall())
print("gold still says:", db.execute("SELECT revenue, orders FROM gold_daily").fetchone(), "(aggregated, not personal, but stale)")
rerun = db.execute("SELECT SUM(amount), COUNT(*) FROM silver_orders").fetchone()
print("after rebuild gold would be:", rerun)

bronze left: 1
silver left: [(11, 'u8')]
log: [('u7', 'bronze_events', 2), ('u7', 'silver_orders', 2)]
gold still says: (150.0, 3) (aggregated, not personal, but stale)
after rebuild gold would be: (70.0, 1)


## Chapter recap

## Practice set

**1.** (easy) A pod requests 250m CPU and runs 8 replicas. How many cores are reserved?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch12.html